## Разведочный анализ данных (EDA)

В этом ноутбуке исследуем датасет об успеваемости студентов и попробуем понять, какие факторы влияют на результаты экзамена по математике.

### Загрузка библиотек и данных

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

# настройка стиля графиков
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

In [ ]:
df = pd.read_csv('data/stud.csv')
df.head()

In [ ]:
print(f'Размер датасета: {df.shape[0]} строк, {df.shape[1]} колонок')

### Описание признаков

Признаки в датасете:
- gender: пол студента
- race/ethnicity: этническая группа (A, B, C, D, E)
- parental level of education: уровень образования родителей
- lunch: тип обеда (standard / free-reduced)
- test preparation course: прошёл ли подготовительный курс
- math score, reading score, writing score: баллы за экзамены

### Базовые проверки данных

In [ ]:
# проверяем пропуски
print('Пропущенные значения:')
print(df.isna().sum())

In [ ]:
# проверяем дубликаты
print(f'Количество дубликатов: {df.duplicated().sum()}')

In [ ]:
df.info()

In [ ]:
# статистика по числовым колонкам
df.describe()

In [ ]:
# уникальные значения категориальных признаков
for col in df.select_dtypes(include='object').columns:
    print(f'{col}: {df[col].nunique()} уникальных значений')
    print(df[col].value_counts())
    print()

### Распределение оценок

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, col in enumerate(['math score', 'reading score', 'writing score']):
    sns.histplot(df[col], kde=True, ax=axes[i], bins=20)
    axes[i].set_title(f'Распределение: {col}')
    axes[i].axvline(df[col].mean(), color='red', linestyle='--', label=f'mean={df[col].mean():.1f}')
    axes[i].legend()

plt.tight_layout()
plt.show()

In [ ]:
# добавим средний балл для удобства анализа
df['average_score'] = (df['math score'] + df['reading score'] + df['writing score']) / 3
print(f'Средний балл по всем предметам: {df["average_score"].mean():.2f}')

### Анализ влияния пола на успеваемость

In [ ]:
# распределение по полу
fig, ax = plt.subplots(figsize=(6, 4))
df['gender'].value_counts().plot(kind='bar', ax=ax, color=['steelblue', 'coral'])
ax.set_title('Распределение по полу')
ax.set_ylabel('Количество')
plt.xticks(rotation=0)
plt.show()

In [ ]:
# сравнение оценок по полу
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for i, col in enumerate(['math score', 'reading score', 'writing score']):
    sns.boxplot(data=df, x='gender', y=col, ax=axes[i])
    axes[i].set_title(col)

plt.tight_layout()
plt.show()

In [ ]:
# средние значения по полу
df.groupby('gender')[['math score', 'reading score', 'writing score']].mean()

Интересно: мужчины в среднем лучше сдают математику, а женщины — чтение и письмо.

### Анализ по этническим группам

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
df['race/ethnicity'].value_counts().sort_index().plot(kind='bar', ax=ax)
ax.set_title('Распределение по этническим группам')
ax.set_ylabel('Количество')
plt.xticks(rotation=0)
plt.show()

In [ ]:
# средние оценки по группам
group_means = df.groupby('race/ethnicity')[['math score', 'reading score', 'writing score']].mean()
group_means.plot(kind='bar', figsize=(10, 5))
plt.title('Средние оценки по этническим группам')
plt.ylabel('Балл')
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.show()

### Влияние образования родителей

In [ ]:
# порядок уровней образования
edu_order = ['some high school', 'high school', 'some college', 
             "associate's degree", "bachelor's degree", "master's degree"]

fig, ax = plt.subplots(figsize=(10, 4))
df['parental level of education'].value_counts()[edu_order].plot(kind='bar', ax=ax)
ax.set_title('Распределение по образованию родителей')
ax.set_ylabel('Количество')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# как образование родителей влияет на оценки
edu_means = df.groupby('parental level of education')['average_score'].mean().reindex(edu_order)

fig, ax = plt.subplots(figsize=(10, 4))
edu_means.plot(kind='bar', ax=ax, color='teal')
ax.set_title('Средний балл в зависимости от образования родителей')
ax.set_ylabel('Средний балл')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

Чем выше образование родителей, тем лучше в среднем успеваемость студентов.

### Влияние типа обеда

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# распределение
df['lunch'].value_counts().plot(kind='pie', ax=axes[0], autopct='%1.1f%%')
axes[0].set_title('Тип обеда')
axes[0].set_ylabel('')

# влияние на оценки
sns.boxplot(data=df, x='lunch', y='average_score', ax=axes[1])
axes[1].set_title('Средний балл по типу обеда')

plt.tight_layout()
plt.show()

In [ ]:
df.groupby('lunch')['average_score'].mean()

Студенты со стандартным обедом показывают заметно лучшие результаты. Возможно, это коррелирует с социально-экономическим статусом семьи.

### Эффект подготовительных курсов

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['test preparation course'].value_counts().plot(kind='pie', ax=axes[0], autopct='%1.1f%%')
axes[0].set_title('Прохождение курсов')
axes[0].set_ylabel('')

sns.boxplot(data=df, x='test preparation course', y='average_score', ax=axes[1])
axes[1].set_title('Влияние курсов на оценки')

plt.tight_layout()
plt.show()

In [ ]:
df.groupby('test preparation course')[['math score', 'reading score', 'writing score']].mean()

Подготовительные курсы дают прирост примерно в 5-6 баллов по каждому предмету.

### Корреляции между оценками

In [ ]:
score_cols = ['math score', 'reading score', 'writing score']
corr = df[score_cols].corr()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, ax=ax, fmt='.2f')
ax.set_title('Корреляции между оценками')
plt.show()

In [ ]:
# pairplot для визуализации зависимостей
sns.pairplot(df, vars=score_cols, hue='gender', height=3)
plt.suptitle('Зависимости между оценками', y=1.02)
plt.show()

Оценки за чтение и письмо сильно коррелируют (0.95), что логично. С математикой корреляция тоже высокая (0.8+), но чуть слабее.

### Проверка выбросов

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for i, col in enumerate(score_cols):
    sns.boxplot(data=df, y=col, ax=axes[i])
    axes[i].set_title(col)

plt.tight_layout()
plt.show()

Выбросов практически нет, данные чистые.

### Выводы

Основные наблюдения из EDA:

1. Данные чистые: нет пропусков, дубликатов и явных выбросов
2. Пол влияет на результаты: мужчины лучше в математике, женщины в чтении/письме
3. Образование родителей положительно коррелирует с успеваемостью
4. Стандартный обед связан с более высокими баллами
5. Подготовительные курсы дают прирост 5-6 баллов
6. Оценки между собой сильно коррелируют

Для предсказания math score имеет смысл использовать все признаки, особенно reading_score и writing_score.